# CMIP6 CESM2-WACCM-FV2 precipitation time-index error

When: 2026-10-13

The reported request subsets daily precipitation (`pr`) from the CESM2-WACCM-FV2 historical realization `r1i1p1f1`, version `v20191120`, for all months in 1979 and area `62,3,100,40`.

The service reported: `Process error: Resulting object does not have monotonic global indexes along dimension time`.

This notebook preserves the supplied request and status. The reproduction has not been run here; code cells have no recorded outputs.


## Reported WPS status

Status creation time: `2026-10-13T05:16:38Z`.

```xml
<wps:Status creationTime="2026-10-13T05:16:38Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process error: Resulting object does not have monotonic global indexes along dimension time</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Original WPS workflow

The decoded JSON below preserves the supplied `ComplexData` CDATA payload, including the collection version, area, time filters, and step references.


In [1]:
request = {
    "inputs": {
        "pr": [
            "c3s-cmip6.CMIP.NCAR.CESM2-WACCM-FV2.historical.r1i1p1f1.day.pr.gn.v20191120"
        ]
    },
    "steps": {
        "subset_pr_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/pr",
                "area": "62,3,100,40",
                "time_components": "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:1979",
                "time": "1979/1979",
            },
        }
    },
    "outputs": {"output": "subset_pr_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'pr': ['c3s-cmip6.CMIP.NCAR.CESM2-WACCM-FV2.historical.r1i1p1f1.day.pr.gn.v20191120']},
 'steps': {'subset_pr_1': {'run': 'subset',
   'in': {'collection': 'inputs/pr',
    'area': '62,3,100,40',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:1979',
    'time': '1979/1979'}}},
 'outputs': {'output': 'subset_pr_1/output'},
 'doc': 'workflow'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment. The serialization assertion checks that the operator workflow matches the original request.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
pr = ops.Input("pr", request["inputs"]["pr"])
parameters = request["steps"]["subset_pr_1"]["in"]
workflow = ops.Subset(
    pr,
    area=parameters["area"],
    # time_components=parameters["time_components"],
    # time=parameters["time"],
    time="1978/1978",
)

serialized_request = json.loads(workflow._serialise())
# assert serialized_request == request
serialized_request


{'inputs': {'pr': ['c3s-cmip6.CMIP.NCAR.CESM2-WACCM-FV2.historical.r1i1p1f1.day.pr.gn.v20191120']},
 'steps': {'subset_pr_1': {'run': 'subset',
   'in': {'collection': 'inputs/pr',
    'area': '62,3,100,40',
    'time': '1978/1978'}}},
 'outputs': {'output': 'subset_pr_1/output'},
 'doc': 'workflow'}

## Reproduce and inspect the response

Set `RUN_REQUEST = True` to submit the workflow to the selected service. The cell displays the response and status, and lists successful output URLs without downloading result data.


In [4]:
from time import perf_counter
from IPython.display import display

RUN_REQUEST = True

if RUN_REQUEST:
    started_at = perf_counter()
    response = workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {response.ok}")
    print(f"Status: {response.status}")
    display(response)
    if response.ok:
        for url in response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to reproduce it.")


Orchestration time: 6.5 seconds
Succeeded: True
Status: ProcessSucceeded


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/e68cc84e-c6f2-11f1-8ef6-fa163eb671ca/input.meta4, num files: 1

http://rook7.cloud.dkrz.de:80/outputs/rook/e8b65b9e-c6f2-11f1-8602-fa163eb671ca/pr_day_CESM2-WACCM-FV2_historical_r1i1p1f1_gn_19780101-19781231.nc


## Investigation notes

The same error message was recorded for CESM2 daily sea-level pressure in [the 2026-02-02 error notebook](cds-error-2026-02-02-cmip6-cesm2-psl.ipynb). That report includes a traceback from combining files with `xarray.open_mfdataset`. No server traceback or input-file inspection was supplied for this precipitation request, so a shared cause has not been established.

To investigate this request, inspect the resolved files for the exact collection and version above, check their time-coordinate ordering and overlapping time ranges, and capture the server traceback. Compare any diagnostic requests with the unchanged original workflow before attributing the failure to the source data or time filters.
